# Predicting 30-Day Hospital Readmissions Among Patients With Diabetes
**Adhilingam Adiswaran | Bellevue University | Milestone 2**

This reproducible notebook tests whether pre-discharge encounter information predicts readmission within 30 days in the UCI **Diabetes 130-US Hospitals (1999–2008)** dataset. It compares Logistic Regression, Random Forest, and (when installed) XGBoost, evaluates patient-grouped holdout and cross-validation performance, and examines feature associations and subgroup disparities.

**Research questions:** (1) How well can encounter data predict early readmission? (2) Do ensembles improve PR-AUC/recall over logistic regression? (3) What features matter? (4) Does performance vary by age, gender, and race?

**Important:** Historical, de-identified educational data; not for clinical decision-making. Associations are not causal effects. The notebook intentionally does not claim performance until it has run.

## 0. Setup
Run top to bottom in a Python 3.10+ Jupyter environment. Install dependencies if necessary:

```bash
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost requests
```

The first data-loading run downloads the official UCI archive if internet is available. If offline, download the archive from https://archive.ics.uci.edu/dataset/296/diabetes+130+us+hospitals+for+years+1999+2008 and put `diabetic_data.csv` or the official ZIP beside this notebook. No patient identifiers are included as model features.

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost ucimlrepo scipy
from pathlib import Path
import io, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay,
                             RocCurveDisplay, brier_score_loss, precision_recall_curve)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibrationDisplay

RANDOM_STATE = 42
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
DATA_URL = 'https://archive.ics.uci.edu/static/public/296/diabetes+130+us+hospitals+for+years+1999+2008.zip'
try:
    from xgboost import XGBClassifier
    HAVE_XGBOOST = True
except ImportError:
    HAVE_XGBOOST = False
    print('XGBoost not installed; run pip install xgboost and restart kernel to include it.')
print('XGBoost available:', HAVE_XGBOOST)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
In C:\Users\adhil\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In C:\Users\adhil\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In C:\Users\adhil\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


XGBoost available: True


## 1. Load and audit the source data
The official UCI ZIP contains `diabetic_data.csv` (and an IDs mapping file). The binary target is `readmitted == '<30'`; `>30` and `NO` are combined as 0. This distinction should be considered when interpreting the outcome.

In [2]:
def load_diabetes():
    for p in [Path('diabetic_data.csv'), Path('data/diabetic_data.csv')]:
        if p.exists():
            print('Loaded local file:', p)
            return pd.read_csv(p, na_values=['?', 'Unknown/Invalid'])
    zip_paths = [Path('diabetes_uci.zip'), Path('data/diabetes_uci.zip')]
    for p in zip_paths:
        if p.exists():
            print('Loaded local ZIP:', p)
            with zipfile.ZipFile(p) as z:
                member = next(x for x in z.namelist() if x.endswith('diabetic_data.csv'))
                return pd.read_csv(z.open(member), na_values=['?', 'Unknown/Invalid'])
    print('Downloading UCI dataset...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=90) as resp:
            raw = resp.read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            member = next(x for x in z.namelist() if x.endswith('diabetic_data.csv'))
            return pd.read_csv(z.open(member), na_values=['?', 'Unknown/Invalid'])
    except Exception as exc:
        raise RuntimeError('Could not download the dataset. Place diabetic_data.csv or the official UCI ZIP in the notebook folder, then rerun.') from exc

df = load_diabetes()
assert {'patient_nbr', 'encounter_id', 'readmitted'}.issubset(df.columns)
print('Rows, columns:', df.shape)
print('Unique patients:', df.patient_nbr.nunique())
display(df.head(3))
display(df['readmitted'].value_counts(dropna=False).to_frame('encounters'))

Loaded local file: diabetic_data.csv


C:\Users\adhil\AppData\Local\Temp\ipykernel_3628\814511729.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(p, na_values=['?', 'Unknown/Invalid'])


Rows, columns: (101766, 50)
Unique patients: 71518


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


,encounters
readmitted,
NO,54864
>30,35545
<30,11357


In [3]:
missing = df.isna().mean().sort_values(ascending=False)
display((100*missing.head(20)).round(1).to_frame('missing_percent'))
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate encounter IDs:', df.encounter_id.duplicated().sum())
print('Patients with multiple encounters:', (df.patient_nbr.value_counts() > 1).sum())
fig, ax = plt.subplots(figsize=(6,4))
df['readmitted'].value_counts().plot.bar(ax=ax)
ax.set(title='Figure 1. Recorded readmission categories', xlabel='Readmission', ylabel='Encounters')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'figure_1_class_balance.png', dpi=160); plt.show()

,missing_percent
weight,96.9
max_glu_serum,94.7
A1Cresult,83.3
medical_specialty,49.1
payer_code,39.6
race,2.2
diag_3,1.4
diag_2,0.4
diag_1,0.0
gender,0.0


Exact duplicate rows: 0
Duplicate encounter IDs: 0
Patients with multiple encounters: 16773


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

Error in callback <function _draw_all_if_interactive at 0x0000025B4E346840> (for post_execute), with arguments args (),kwargs {}:


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

<Figure size 600x400 with 1 Axes>

## 2. Define outcome, dictionary, and reproducible preprocessing
**Data dictionary (selected fields):** `patient_nbr` = patient grouping ID (never a predictor); `encounter_id` = encounter ID (never a predictor); `readmitted` = original three-category outcome; `target_30d` = 1 if `<30`, else 0; `age` = age band; `gender`, `race` = recorded demographic categories; `time_in_hospital` = stay length; `num_lab_procedures`, `num_procedures`, `num_medications` = counts; `number_outpatient`, `number_emergency`, `number_inpatient` = prior utilization counts; `A1Cresult`, `max_glu_serum` = lab indicators; `change`, `diabetesMed` = medication indicators.

To limit leakage and complexity, use a documented subset of plausible pre-discharge variables; exclude encounter/patient IDs and the outcome. `payer_code`, `medical_specialty`, diagnosis codes, and discharge disposition are not modeled in this baseline because of high missingness, complexity, or uncertain prediction-time availability. **Important:** The chosen feature set includes information known by discharge, not necessarily at admission. Imputation and encoding are fitted only within training folds.

In [4]:
numeric_cols = ['time_in_hospital','num_lab_procedures','num_procedures','num_medications',
                'number_outpatient','number_emergency','number_inpatient','number_diagnoses']
categorical_cols = ['age','gender','race','admission_type_id','admission_source_id',
                    'A1Cresult','max_glu_serum','change','diabetesMed','insulin']
feature_cols = numeric_cols + categorical_cols
missing_features = sorted(set(feature_cols) - set(df.columns))
assert not missing_features, f'Missing expected columns: {missing_features}'
work = df.drop_duplicates(subset=['encounter_id']).copy()
work['target_30d'] = work['readmitted'].eq('<30').astype(int)
for c in numeric_cols:
    work[c] = pd.to_numeric(work[c], errors='coerce')
for c in categorical_cols:
    work[c] = work[c].astype('object').where(work[c].notna(), np.nan)
X = work[feature_cols].copy()
y = work['target_30d'].copy()
groups = work['patient_nbr'].copy()
print('Encounter count:',len(work),'positive prevalence:',round(y.mean(),4))
print('Excluded columns:',sorted(set(df.columns)-set(feature_cols)-{'readmitted'}))

def make_preprocessor():
    num = Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
    cat = Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                    ('onehot',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric',num,numeric_cols),('categorical',cat,categorical_cols)])

Encounter count: 101766 positive prevalence: 0.1116
Excluded columns: ['acarbose', 'acetohexamide', 'chlorpropamide', 'citoglipton', 'diag_1', 'diag_2', 'diag_3', 'discharge_disposition_id', 'encounter_id', 'examide', 'glimepiride', 'glimepiride-pioglitazone', 'glipizide', 'glipizide-metformin', 'glyburide', 'glyburide-metformin', 'medical_specialty', 'metformin', 'metformin-pioglitazone', 'metformin-rosiglitazone', 'miglitol', 'nateglinide', 'patient_nbr', 'payer_code', 'pioglitazone', 'repaglinide', 'rosiglitazone', 'tolazamide', 'tolbutamide', 'troglitazone', 'weight']


## 3. Patient-disjoint holdout split
A patient can have multiple encounters. A random encounter split would leak patient history across training and testing. `GroupShuffleSplit` reserves approximately 20% of patients for holdout; `StratifiedGroupKFold` groups patients during cross-validation while approximately balancing class proportions. The holdout set is never used for fitting or hyperparameter selection.

In [5]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]
assert set(g_train).isdisjoint(set(g_test))
print('Training:',len(y_train),'Test:',len(y_test))
print('Training prevalence:',round(y_train.mean(),4),'Test prevalence:',round(y_test.mean(),4))
print('Patient overlap:',len(set(g_train)&set(g_test)))

Training: 81613 Test: 20153
Training prevalence: 0.1128 Test prevalence: 0.1067
Patient overlap: 0


## 4. Methods and justification
**Logistic Regression:** interpretable, regularized baseline with class weighting. **Random Forest:** nonlinear interactions and robustness to complex feature relationships. **XGBoost:** gradient-boosted trees often effective for tabular clinical records; optional dependency. Class imbalance is addressed using training-derived class weights (or `scale_pos_weight` for XGBoost), not resampling across patients. **Primary metric:** PR-AUC (average precision), which is informative for minority positive classes. Also report ROC-AUC, recall, precision, F1, Brier score and confusion matrices. The default 0.5 threshold is illustrative, not clinically optimized. No model is claimed superior until metrics are calculated.

In [14]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Generate predictions
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.6730015382325212
Precision: 0.1586959864677841
Recall: 0.4797768479776848
F1 Score: 0.2385024266235267
ROC-AUC: 0.6268659466578201
PR-AUC: 0.17906248906875505

Confusion Matrix:
[[12531  5471]
 [ 1119  1032]]


## 5. Validation figures
Figures 2–4 show holdout precision-recall curves, confusion matrices, and calibration. All figures and tables are exported to `outputs/` for the white paper. Because class weights change effective training prevalence, predicted probabilities may be poorly calibrated; inspect the calibration plot before interpreting scores as risks.

In [19]:
from sklearn.metrics import classification_report

y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.70      0.79     18002
           1       0.16      0.48      0.24      2151

    accuracy                           0.67     20153
   macro avg       0.54      0.59      0.52     20153
weighted avg       0.84      0.67      0.73     20153



## 6. Model interpretation
Permutation importance measures how much shuffling a feature reduces holdout PR-AUC. It captures predictive association, **not causal influence**. To avoid excessive computation, use a fixed random sample of up to 5,000 holdout encounters for the exploratory plot. Since this uses holdout data for explanation, do not use these importances to retune the model without a new untouched test set.

In [28]:

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate
)

# Patient-level groups are required
if "g_train" not in globals():
    raise NameError(
        "g_train is missing. Patient groups are required."
    )

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

cv_rows = []

for name, trained_pipe in fitted.items():
    print("Cross-validating:", name)

    scores = cross_validate(
        trained_pipe,
        X_train,
        y_train,
        groups=g_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise"
    )

    row = {"model": name}

    for metric in scoring:
        row[f"cv_{metric}_mean"] = np.mean(
            scores[f"test_{metric}"]
        )
        row[f"cv_{metric}_std"] = np.std(
            scores[f"test_{metric}"]
        )

    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index("model")

display(cv_results.round(4))
best_name = cv_results["cv_pr_auc_mean"].idxmax()
best_pipe = fitted[best_name]

print("Best model:", best_name)


Cross-validating: Logistic Regression
Cross-validating: Random Forest


,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_recall_mean,cv_recall_std,cv_precision_mean,cv_precision_std,cv_f1_mean,cv_f1_std
model,,,,,,,,,,
Logistic Regression,0.1945,0.0088,0.6325,0.0102,0.4940,0.0175,0.1668,0.0045,0.2494,0.0071
Random Forest,0.1575,0.0025,0.6012,0.0041,0.0481,0.0031,0.2326,0.0085,0.0797,0.0044


Best model: Logistic Regression


## 7. Fairness / subgroup audit
Evaluate recorded age, gender and race categories on the **same held-out encounters**. Small groups have unstable metrics; counts are shown and results with fewer than 30 positive cases should not be interpreted as reliable. Historical administrative demographic fields are imperfect proxies for lived identity. These metrics are descriptive, not evidence of clinical fairness or efficacy.

In [31]:
probas = {}

for name, model in fitted.items():
    probas[name] = model.predict_proba(X_test)[:, 1]
    print(f"Probabilities generated for: {name}")

print("\nAvailable models:", list(probas.keys()))
p = probas[best_name]
audit = work.iloc[test_idx][['age','gender','race']].copy().reset_index(drop=True)
audit['truth']=y_test.to_numpy();audit['probability']=p
audit['prediction']=(p>=0.5).astype(int)
rows=[]
for attribute in ['age','gender','race']:
    for value, sub in audit.groupby(attribute,dropna=False):
        npos=int(sub.truth.sum())
        rows.append({'attribute':attribute,'group':str(value),'n':len(sub),'positive_n':npos,
                     'prevalence':sub.truth.mean(),
                     'recall':recall_score(sub.truth,sub.prediction,zero_division=0) if npos else np.nan,
                     'precision':precision_score(sub.truth,sub.prediction,zero_division=0),
                     'pr_auc':average_precision_score(sub.truth,sub.probability) if npos else np.nan,
                     'false_negative_n':int(((sub.truth==1)&(sub.prediction==0)).sum()),
                     'caution_small_positive_n':npos<30})
subgroup = pd.DataFrame(rows).sort_values(['attribute','group'])
display(subgroup.round(3))
subgroup.to_csv(OUTPUT_DIR/'subgroup_metrics.csv',index=False)
print('Groups with fewer than 30 positives:',int(subgroup.caution_small_positive_n.sum()))

Probabilities generated for: Logistic Regression
Probabilities generated for: Random Forest

Available models: ['Logistic Regression', 'Random Forest']


,attribute,group,n,positive_n,prevalence,recall,precision,pr_auc,false_negative_n,caution_small_positive_n
0,age,[0-10),38,2,0.053,0.000,0.000,0.054,2,True
1,age,[10-20),137,5,0.036,0.000,0.000,0.070,5,True
2,age,[20-30),285,23,0.081,0.304,0.115,0.118,16,True
3,age,[30-40),804,99,0.123,0.414,0.198,0.260,58,False
4,age,[40-50),1893,193,0.102,0.430,0.196,0.216,110,False
5,age,[50-60),3518,356,0.101,0.346,0.215,0.190,233,False
6,age,[60-70),4511,465,0.103,0.529,0.159,0.174,219,False
7,age,[70-80),5125,560,0.109,0.511,0.138,0.175,274,False
8,age,[80-90),3314,403,0.122,0.563,0.159,0.176,176,False
9,age,[90-100),528,45,0.085,0.422,0.101,0.098,26,False


Groups with fewer than 30 positives: 7


## 8. Interpretation and conclusions (complete after execution)
Use the saved tables to report: (1) holdout PR-AUC against the positive-class prevalence baseline; (2) cross-validation means and variability; (3) recall/precision trade-offs at threshold 0.5; (4) calibration; (5) subgroup sizes and disparities. The model selected here uses **training CV PR-AUC**, not holdout PR-AUC. A higher ensemble PR-AUC supports the working hypothesis only on this historical dataset. The historical labels and absence of external validation prohibit clinical deployment.

**Assumptions:** UCI labels and recorded fields are sufficiently consistent for retrospective analysis; all modeled variables are available by discharge; patient IDs reliably group encounters.

**Limitations:** 1999–2008 data, possible unobserved confounding and label ambiguity, nonrepresentative hospitals, potential leakage from encounter timing, imbalanced outcome, and uncalibrated probabilities.

**Ethical risks and mitigation:** Privacy (high impact): exclude IDs and avoid re-identification; bias (high impact): subgroup audit and minimum counts; leakage (high impact): grouped split and fold-only transforms; drift (high likelihood): contemporary external validation before use; misuse (high impact): no autonomous discharge decisions.

**Next steps:** Validate on contemporary multi-site data, check feature availability at discharge, tune thresholds using prospective costs, calibrate scores, conduct clinician review and impact assessment, and monitor drift/fairness. Never use this academic prototype to guide patient care.

## 9. Reproducibility / references
- Dataset: Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). *Diabetes 130-US hospitals for years 1999–2008* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5230J
- Study: Strack, B., et al. (2014). Impact of HbA1c measurement on hospital readmission rates: Analysis of 70,000 clinical database patient records. *BioMed Research International, 2014*, Article 781670. https://doi.org/10.1155/2014/781670

**Outputs:** five PNG figures plus CSV files with CV, holdout, subgroup, and importance results. Re-run every cell after changing packages or the dataset. Runtime can be several minutes, depending on hardware. For publication, record package versions and execution date.

In [32]:
import sklearn, sys
print('Python:',sys.version.split()[0], 'pandas:',pd.__version__, 'scikit-learn:',sklearn.__version__)
print('Selected by training CV PR-AUC:',best_name)
print('Holdout positive prevalence (no-skill AP baseline):',round(y_test.mean(),4))
print('Saved artifacts:')
for pth in sorted(OUTPUT_DIR.iterdir()):print(' -',pth)

Python: 3.13.5 pandas: 2.3.3 scikit-learn: 1.9.0
Selected by training CV PR-AUC: Logistic Regression
Holdout positive prevalence (no-skill AP baseline): 0.1067
Saved artifacts:
 - outputs\figure_2_precision_recall.png
 - outputs\subgroup_metrics.csv
